In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [2]:
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Evaluation Metrics

**1. Perplexity**
- Measures how well model predicts test data
- Lower is better
- Formula: PPL = exp(cross_entropy_loss)

In [ ]:
def calculate_perplexity(model, tokenizer, text):
    encodings = tokenizer(text, return_tensors="pt")
    max_length = model.config.n_positions
    stride = 512
    
    nlls = []
    for i in range(0, encodings.input_ids.size(1), stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc = min(i + stride, encodings.input_ids.size(1))
        trg_len = end_loc - i
        
        input_ids = encodings.input_ids[:, begin_loc:end_loc]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        
        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss * trg_len
        
        nlls.append(neg_log_likelihood)
    
    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    return ppl.item()

**2. Task-Specific Metrics**
- Classification: Accuracy, F1-score, Precision, Recall
- Generation: BLEU, ROUGE, METEOR
- Question Answering: Exact Match, F1-score
- Summarization: ROUGE-L, BLEU


**3. Human Evaluation**
- Quality: Fluency, coherence, relevance
- Safety: Harmful content, bias
- Helpfulness: Task completion, correctness

### Standard Benchmarks

1. GLUE (General Language Understanding Evaluation)
- 9 tasks: sentiment, paraphrase, inference, etc.
- Measures general language understanding


2. SuperGLUE
- More challenging version of GLUE
- Includes reading comprehension, reasoning

**3. MMLU (Massive Multitask Language Understanding)**
- 57 tasks across STEM, humanities, social sciences
- Measures broad knowledge and reasoning

In [4]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [5]:
mmlu = load_dataset("cais/mmlu", "all", split="test")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

all/auxiliary_train-00000-of-00001.parqu(…):   0%|          | 0.00/47.5M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [ ]:
def evaluate_mmlu(model, tokenizer, dataset, subject="all"):
    correct = 0
    total = 0
    
    for example in dataset:
        if subject != "all" and example["subject"] != subject:
            continue
        
        prompt = f"{example['question']}\nA. {example['choices'][0]}\nB. {example['choices'][1]}\nC. {example['choices'][2]}\nD. {example['choices'][3]}\nAnswer:"
        
        inputs = tokenizer(prompt, return_tensors="pt")
        outputs = model.generate(**inputs, max_new_tokens=1)
        answer = tokenizer.decode(outputs[0][-1])
        
        correct_answer = example['choices'][example['answer']]
        if answer.strip().upper() == correct_answer[0]:
            correct += 1
        total += 1
    
    accuracy = correct / total if total > 0 else 0
    return accuracy

In [ ]:
accuracy = evaluate_mmlu(model, tokenizer, mmlu, subject="all")
print(f"MMLU Accuracy: {accuracy:.2%}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

4. HELM (Holistic Evaluation of Language Models)
- Comprehensive evaluation across many scenarios
- Includes accuracy, calibration, robustness, fairness

5. BIG-bench (Beyond the Imitation Game)
- 200+ diverse tasks
- Tests reasoning, knowledge, creativity

6. HumanEval
- Code generation benchmark
- Measures functional correctness

7. TruthfulQA
- Measures truthfulness and avoiding falsehoods
- Important for safety evaluation